# IT4060 - HPC Failure Prediction

## Notebook: 24h Model Finalization

This notebook finalizes the modeling step on the `label_next_24h` target by tuning the two strongest candidate families, selecting the winner on validation, and exporting final report-ready artifacts.

In [1]:
from pathlib import Path
import os
import time

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 220)
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (14, 6)


In [2]:
PROJECT_NAME = 'IT4060-ML-Assignment-HPC-Failure-Prediction'
TARGET_COLUMN = 'label_next_24h'
MODEL_DISPLAY_NAMES = {
    'extra_trees': 'Extra Trees',
    'hist_gradient_boosting': 'Hist Gradient Boosting',
}
EXTRA_TREES_CONFIGS = [
    {'config_name': 'et_base', 'n_estimators': 200, 'max_depth': 18, 'min_samples_leaf': 5, 'max_features': 'sqrt'},
    {'config_name': 'et_regularized', 'n_estimators': 300, 'max_depth': 14, 'min_samples_leaf': 10, 'max_features': 'sqrt'},
]
HGB_CONFIGS = [
    {'config_name': 'hgb_base', 'learning_rate': 0.05, 'max_iter': 200, 'max_depth': 6, 'max_leaf_nodes': 31, 'min_samples_leaf': 50, 'l2_regularization': 0.1},
    {'config_name': 'hgb_deeper', 'learning_rate': 0.03, 'max_iter': 300, 'max_depth': 8, 'max_leaf_nodes': 63, 'min_samples_leaf': 30, 'l2_regularization': 0.5},
]

def find_project_root():
    cwd = Path.cwd().resolve()
    home = Path.home().resolve()
    desktop = home / 'Desktop'
    candidate_roots = [cwd, *cwd.parents, home, desktop, desktop / 'Manilka' / 'ML_Assignment']
    seen = set()

    for base in candidate_roots:
        for candidate in (base, base / PROJECT_NAME):
            if candidate in seen or not candidate.exists():
                continue
            seen.add(candidate)
            if (candidate / 'data' / 'processed').exists() and (candidate / 'results').exists():
                return candidate

    raise FileNotFoundError('Could not locate the project root with data/processed and results/.')

project_root = find_project_root()
processed_path = project_root / 'data' / 'processed' / 'node_hour_features_multi_horizon.csv.gz'
results_dir = project_root / 'results'
final_results_dir = results_dir / 'model_finalization_24h'
final_results_dir.mkdir(parents=True, exist_ok=True)

label_summary_path = final_results_dir / 'label_distribution.csv'
validation_tuning_path = final_results_dir / 'validation_tuning_results.csv'
selected_models_path = final_results_dir / 'selected_models.csv'
test_results_path = final_results_dir / 'selected_model_test_results.csv'
top_risk_path = final_results_dir / 'selected_top_risk_rows.csv'
feature_importance_path = final_results_dir / 'best_extra_trees_feature_importance.csv'
validation_plot_path = final_results_dir / 'validation_tuning_overview.png'
test_plot_path = final_results_dir / 'selected_model_test_metrics.png'
threshold_plot_path = final_results_dir / 'selected_model_threshold_tradeoff.png'

print(f'Project root: {project_root}')
print(f'Processed data path: {processed_path}')
print(f'Finalization results directory: {final_results_dir}')


Project root: D:\Projects\IT4060-ML-Assignment-HPC-Failure-Prediction
Processed data path: D:\Projects\IT4060-ML-Assignment-HPC-Failure-Prediction\data\processed\node_hour_features_multi_horizon.csv.gz
Finalization results directory: D:\Projects\IT4060-ML-Assignment-HPC-Failure-Prediction\results\model_finalization_24h


In [3]:
df = pd.concat(
    pd.read_csv(
        processed_path,
        compression='gzip',
        low_memory=False,
        chunksize=100000,
    ),
    ignore_index=True,
)
df['hour'] = pd.to_datetime(df['hour'])
df['next_failure_time'] = pd.to_datetime(df['next_failure_time'])

target_columns = [column for column in df.columns if column.startswith('label_next_')]
feature_exclusions = ['hour', 'next_failure_time', 'hours_to_next_failure', *target_columns]
feature_columns = [column for column in df.columns if column not in feature_exclusions]

unique_hours = np.sort(df['hour'].unique())
train_end = int(len(unique_hours) * 0.70)
valid_end = int(len(unique_hours) * 0.85)

train_hours = unique_hours[:train_end]
valid_hours = unique_hours[train_end:valid_end]
test_hours = unique_hours[valid_end:]

train_df = df[df['hour'].isin(train_hours)].copy()
valid_df = df[df['hour'].isin(valid_hours)].copy()
test_df = df[df['hour'].isin(test_hours)].copy()

label_summary = pd.DataFrame([
    {"split": "train", "rows": len(train_df), "positives": int(train_df[TARGET_COLUMN].sum()), "positive_rate": float(train_df[TARGET_COLUMN].mean())},
    {"split": "validation", "rows": len(valid_df), "positives": int(valid_df[TARGET_COLUMN].sum()), "positive_rate": float(valid_df[TARGET_COLUMN].mean())},
    {"split": "test", "rows": len(test_df), "positives": int(test_df[TARGET_COLUMN].sum()), "positive_rate": float(test_df[TARGET_COLUMN].mean())},
])
label_summary.to_csv(label_summary_path, index=False)
display(label_summary)


,split,rows,positives,positive_rate
0,train,488782,2358,0.004824
1,validation,114395,305,0.002666
2,test,84637,157,0.001855


In [4]:
def precision_at_k(y_true, scores, k=50):
    if len(scores) == 0:
        return np.nan
    k = min(k, len(scores))
    top_indices = np.argsort(scores)[::-1][:k]
    return float(np.asarray(y_true)[top_indices].mean())

def evaluate_scores(y_true, scores, threshold):
    predictions = (scores >= threshold).astype(int)
    return {
        'pr_auc': average_precision_score(y_true, scores),
        'roc_auc': roc_auc_score(y_true, scores),
        'precision': precision_score(y_true, predictions, zero_division=0),
        'recall': recall_score(y_true, predictions, zero_division=0),
        'f1': f1_score(y_true, predictions, zero_division=0),
        'predicted_positives': int(predictions.sum()),
        'precision_at_50': precision_at_k(y_true, scores, k=50),
    }

def build_threshold_outputs(y_true, scores, default_threshold=0.5):
    precision_curve, recall_curve, threshold_curve = precision_recall_curve(y_true, scores)
    f1_curve = 2 * precision_curve[:-1] * recall_curve[:-1] / np.clip(precision_curve[:-1] + recall_curve[:-1], 1e-12, None)
    best_threshold = float(threshold_curve[np.nanargmax(f1_curve)]) if len(threshold_curve) else default_threshold
    metrics_df = pd.DataFrame([
        {"threshold_strategy": "default", "threshold": default_threshold, **evaluate_scores(y_true, scores, default_threshold)},
        {"threshold_strategy": "best_validation", "threshold": best_threshold, **evaluate_scores(y_true, scores, best_threshold)},
    ])
    return metrics_df, best_threshold

def run_extra_trees(config):
    X_train = train_df[feature_columns]
    y_train = train_df[TARGET_COLUMN]
    X_valid = valid_df[feature_columns]
    y_valid = valid_df[TARGET_COLUMN]
    X_test = test_df[feature_columns]
    y_test = test_df[TARGET_COLUMN]

    categorical_features = ['node']
    numeric_features = [column for column in feature_columns if column not in categorical_features]
    preprocessor = ColumnTransformer(
        transformers=[
            ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
            ('numeric', Pipeline(steps=[('imputer', SimpleImputer(strategy='constant', fill_value=0))]), numeric_features),
        ]
    )
    model_pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', ExtraTreesClassifier(
                n_estimators=config['n_estimators'],
                max_depth=config['max_depth'],
                min_samples_leaf=config['min_samples_leaf'],
                max_features=config['max_features'],
                class_weight='balanced',
                n_jobs=1,
                random_state=42,
                verbose=0,
            )),
        ]
    )
    start = time.perf_counter()
    model_pipeline.fit(X_train, y_train)
    training_seconds = time.perf_counter() - start
    valid_scores = model_pipeline.predict_proba(X_valid)[:, 1]
    test_scores = model_pipeline.predict_proba(X_test)[:, 1]
    validation_metrics, best_threshold = build_threshold_outputs(y_valid, valid_scores, default_threshold=0.5)
    test_metrics, _ = build_threshold_outputs(y_test, test_scores, default_threshold=0.5)
    test_metrics.loc[test_metrics["threshold_strategy"] == "best_validation", "threshold"] = best_threshold
    feature_importance = pd.DataFrame({
        "feature": model_pipeline.named_steps["preprocessor"].get_feature_names_out(),
        "importance": model_pipeline.named_steps["model"].feature_importances_,
    }).sort_values("importance", ascending=False)
    return model_pipeline, feature_importance, training_seconds, validation_metrics, test_metrics

def run_hist_gradient_boosting(config):
    X_train = train_df[feature_columns].copy()
    y_train = train_df[TARGET_COLUMN]
    X_valid = valid_df[feature_columns].copy()
    y_valid = valid_df[TARGET_COLUMN]
    X_test = test_df[feature_columns].copy()
    y_test = test_df[TARGET_COLUMN]
    imputer = SimpleImputer(strategy='median')
    X_train.loc[:, feature_columns] = imputer.fit_transform(X_train[feature_columns])
    X_valid.loc[:, feature_columns] = imputer.transform(X_valid[feature_columns])
    X_test.loc[:, feature_columns] = imputer.transform(X_test[feature_columns])
    model = HistGradientBoostingClassifier(
        loss='log_loss',
        learning_rate=config['learning_rate'],
        max_iter=config['max_iter'],
        max_leaf_nodes=config['max_leaf_nodes'],
        max_depth=config['max_depth'],
        min_samples_leaf=config['min_samples_leaf'],
        l2_regularization=config['l2_regularization'],
        early_stopping=False,
        class_weight='balanced',
        random_state=42,
        verbose=0,
    )
    start = time.perf_counter()
    model.fit(X_train, y_train)
    training_seconds = time.perf_counter() - start
    valid_scores = model.predict_proba(X_valid)[:, 1]
    test_scores = model.predict_proba(X_test)[:, 1]
    validation_metrics, best_threshold = build_threshold_outputs(y_valid, valid_scores, default_threshold=0.5)
    test_metrics, _ = build_threshold_outputs(y_test, test_scores, default_threshold=0.5)
    test_metrics.loc[test_metrics["threshold_strategy"] == "best_validation", "threshold"] = best_threshold
    return model, training_seconds, validation_metrics, test_metrics


In [ ]:
tuning_rows = []
best_models = {}

for config in EXTRA_TREES_CONFIGS:
    print(f"Training Extra Trees config: {config['config_name']}")
    model_pipeline, feature_importance, training_seconds, validation_metrics, test_metrics = run_extra_trees(config)
    validation_default = validation_metrics[validation_metrics["threshold_strategy"] == "default"].iloc[0].to_dict()
    tuning_rows.append({
        'model_name': 'extra_trees',
        'display_name': MODEL_DISPLAY_NAMES['extra_trees'],
        'config_name': config['config_name'],
        **config,
        "training_seconds": training_seconds,
        **validation_default,
    })
    selection_key = (validation_default["pr_auc"], validation_default["precision_at_50"], validation_default["roc_auc"])
    if "extra_trees" not in best_models or selection_key > best_models["extra_trees"]["selection_key"]:
        best_models["extra_trees"] = {
            "selection_key": selection_key,
            "config": config,
            "training_seconds": training_seconds,
            "validation_metrics": validation_metrics.copy(),
            "test_metrics": test_metrics.copy(),
            "feature_importance": feature_importance.copy(),
        }

for config in HGB_CONFIGS:
    print(f"Training Hist Gradient Boosting config: {config['config_name']}")
    model, training_seconds, validation_metrics, test_metrics = run_hist_gradient_boosting(config)
    validation_default = validation_metrics[validation_metrics["threshold_strategy"] == "default"].iloc[0].to_dict()
    tuning_rows.append({
        'model_name': 'hist_gradient_boosting',
        'display_name': MODEL_DISPLAY_NAMES['hist_gradient_boosting'],
        'config_name': config['config_name'],
        **config,
        "training_seconds": training_seconds,
        **validation_default,
    })
    selection_key = (validation_default["pr_auc"], validation_default["precision_at_50"], validation_default["roc_auc"])
    if "hist_gradient_boosting" not in best_models or selection_key > best_models["hist_gradient_boosting"]["selection_key"]:
        best_models["hist_gradient_boosting"] = {
            "selection_key": selection_key,
            "config": config,
            "training_seconds": training_seconds,
            "validation_metrics": validation_metrics.copy(),
            "test_metrics": test_metrics.copy(),
        }

validation_tuning = pd.DataFrame(tuning_rows)
validation_tuning.to_csv(validation_tuning_path, index=False)
display(validation_tuning.sort_values(["model_name", "pr_auc"], ascending=[True, False]))


Training Extra Trees config: et_base
Training Extra Trees config: et_regularized


,model_name,display_name,config_name,n_estimators,max_depth,min_samples_leaf,max_features,training_seconds,threshold_strategy,threshold,pr_auc,roc_auc,precision,recall,f1,predicted_positives,precision_at_50,learning_rate,max_iter,max_leaf_nodes,l2_regularization
1,extra_trees,Extra Trees,et_regularized,300.0,14,10,sqrt,205.091770,default,0.5,0.010834,0.548962,0.003420,0.403279,0.006782,35968,0.04,NaN,NaN,NaN,NaN
0,extra_trees,Extra Trees,et_base,200.0,18,5,sqrt,136.808630,default,0.5,0.009175,0.544389,0.003445,0.396721,0.006830,35125,0.06,NaN,NaN,NaN,NaN
2,hist_gradient_boosting,Hist Gradient Boosting,hgb_base,NaN,6,50,NaN,45.401412,default,0.5,0.009820,0.496431,0.003247,0.163934,0.006367,15401,0.12,0.05,200.0,31.0,0.1
3,hist_gradient_boosting,Hist Gradient Boosting,hgb_deeper,NaN,8,30,NaN,86.640437,default,0.5,0.008530,0.491660,0.004346,0.137705,0.008427,9663,0.10,0.03,300.0,63.0,0.5


In [ ]:
selected_rows = []
selected_test_frames = []
top_risk_rows = []

for model_name, payload in best_models.items():
    validation_default = payload["validation_metrics"][payload["validation_metrics"]["threshold_strategy"] == "default"].iloc[0]
    selected_rows.append({
        "model_name": model_name,
        "display_name": MODEL_DISPLAY_NAMES[model_name],
        "config_name": payload["config"]["config_name"],
        "training_seconds": payload["training_seconds"],
        "validation_pr_auc": validation_default["pr_auc"],
        "validation_roc_auc": validation_default["roc_auc"],
        "validation_precision_at_50": validation_default["precision_at_50"],
    })

    test_df_metrics = payload["test_metrics"].copy()
    test_df_metrics.insert(0, "model_name", model_name)
    test_df_metrics.insert(1, "display_name", MODEL_DISPLAY_NAMES[model_name])
    test_df_metrics.insert(2, "config_name", payload["config"]["config_name"])
    selected_test_frames.append(test_df_metrics)

selected_models = pd.DataFrame(selected_rows).sort_values(["validation_pr_auc", "validation_precision_at_50", "validation_roc_auc"], ascending=False)
selected_models.to_csv(selected_models_path, index=False)
selected_test_results = pd.concat(selected_test_frames, ignore_index=True)
selected_test_results.to_csv(test_results_path, index=False)

overall_winner = selected_models.iloc[0]["model_name"]
if overall_winner == "extra_trees":
    best_models["extra_trees"]["feature_importance"].to_csv(feature_importance_path, index=False)

display(selected_models)
display(selected_test_results)


,model_name,display_name,config_name,training_seconds,validation_pr_auc,validation_roc_auc,validation_precision_at_50
0,extra_trees,Extra Trees,et_regularized,205.091770,0.010834,0.548962,0.04
1,hist_gradient_boosting,Hist Gradient Boosting,hgb_base,45.401412,0.009820,0.496431,0.12


,model_name,display_name,config_name,threshold_strategy,threshold,pr_auc,roc_auc,precision,recall,f1,predicted_positives,precision_at_50
0,extra_trees,Extra Trees,et_regularized,default,0.500000,0.008593,0.606745,0.003157,0.439490,0.006270,21854,0.0
1,extra_trees,Extra Trees,et_regularized,best_validation,0.783822,0.008593,0.606745,0.036620,0.165605,0.059977,710,0.0
2,hist_gradient_boosting,Hist Gradient Boosting,hgb_base,default,0.500000,0.009062,0.593044,0.004716,0.292994,0.009284,9753,0.0
3,hist_gradient_boosting,Hist Gradient Boosting,hgb_base,best_validation,0.929347,0.009062,0.593044,0.050891,0.127389,0.072727,393,0.0


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
for axis, metric in zip(axes, ['pr_auc', 'roc_auc', 'precision_at_50']):
    sns.barplot(data=validation_tuning, x='config_name', y=metric, hue='display_name', palette='viridis', ax=axis)
    axis.set_title(f"Validation {metric.replace('_', ' ').upper()}")
    axis.set_xlabel('Config')
    axis.set_ylabel(metric.replace('_', ' ').upper())
    axis.tick_params(axis='x', rotation=20)
plt.tight_layout()
fig.savefig(validation_plot_path, bbox_inches='tight')
plt.show()

default_test = selected_test_results[selected_test_results['threshold_strategy'] == 'default']
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
for axis, metric in zip(axes, ['pr_auc', 'roc_auc', 'precision_at_50']):
    sns.barplot(data=default_test, x='display_name', y=metric, hue='display_name', dodge=False, legend=False, palette='magma', ax=axis)
    axis.set_title(f"Test {metric.replace('_', ' ').upper()} for Selected Models")
    axis.set_xlabel('Model')
    axis.set_ylabel(metric.replace('_', ' ').upper())
plt.tight_layout()
fig.savefig(test_plot_path, bbox_inches='tight')
plt.show()

tradeoff_df = selected_test_results[selected_test_results['threshold_strategy'].isin(['default', 'best_validation'])].copy()
metric_df = tradeoff_df.melt(id_vars=['display_name', 'threshold_strategy'], value_vars=['precision', 'recall', 'f1'], var_name='metric', value_name='value')
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
for axis, metric in zip(axes, ['precision', 'recall', 'f1']):
    subset = metric_df[metric_df['metric'] == metric]
    sns.barplot(data=subset, x='display_name', y='value', hue='threshold_strategy', palette='rocket', ax=axis)
    axis.set_title(f"Selected Model {metric.upper()} by Threshold Strategy")
    axis.set_xlabel('Model')
    axis.set_ylabel(metric.upper())
plt.tight_layout()
fig.savefig(threshold_plot_path, bbox_inches='tight')
plt.show()


## Summary

This notebook finalizes the `label_next_24h` modeling stage after [10_horizon_comparison.ipynb](10_horizon_comparison.ipynb) showed that wider horizons are more learnable than `6h`. The key outputs are the validation tuning table, the selected-model test table, and the report plots in `results/model_finalization_24h/`, because they make the final model choice and operating-threshold trade-off explicit.

Persisted finalization artifacts are written to `results/model_finalization_24h/`.
